# Homework 13

## Импорты, seed и среда

In [1]:

# Базовые библиотеки для воспроизводимости, анализа и удобного отображения результатов.
import random
from typing import Iterable

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from datasets import load_dataset
from collections import Counter

/Users/polina/vscode/art_ing_course/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)

print(f"Device: {device}")

Device: mps


In [3]:
# Берём стандартную мультиязычную BERT-модель: она хорошо подходит для русскоязычных примеров.
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model max length:", tokenizer.model_max_length)

Tokenizer loaded: bert-base-multilingual-cased
Tokenizer class: BertTokenizer
Model max length: 512


## Датасет и первичный анализ

In [4]:
dataset = load_dataset("dair-ai/emotion")

In [5]:
for split_name, split_data in dataset.items():
    print(f"{split_name:12s}: {len(split_data):>6} примеров")

train       :  16000 примеров
validation  :   2000 примеров
test        :   2000 примеров


In [6]:
label_names = dataset["train"].features["label"].names
print("Классы:", label_names)

Классы: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [7]:
train_labels = dataset["train"]["label"]
counts = Counter(train_labels)
print("\nРаспределение классов (train):")
for idx, count in sorted(counts.items()):
    print(f"  {label_names[idx]:10s}: {count:>5} ({count/len(train_labels)*100:.1f}%)")


Распределение классов (train):
  sadness   :  4666 (29.2%)
  joy       :  5362 (33.5%)
  love      :  1304 (8.2%)
  anger     :  2159 (13.5%)
  fear      :  1937 (12.1%)
  surprise  :   572 (3.6%)


In [8]:
print("\n 5 примеров из train ")
for i in range(5):
    ex = dataset["train"][i]
    print(f"  [{label_names[ex['label']]:8s}] {ex['text'][:80]}")

print("\n 3 примера из validation ")
for i in range(3):
    ex = dataset["validation"][i]
    print(f"  [{label_names[ex['label']]:8s}] {ex['text'][:80]}")


 5 примеров из train 
  [sadness ] i didnt feel humiliated
  [sadness ] i can go from feeling so hopeless to so damned hopeful just from being around so
  [anger   ] im grabbing a minute to post i feel greedy wrong
  [love    ] i am ever feeling nostalgic about the fireplace i will know that it is still on 
  [anger   ] i am feeling grouchy

 3 примера из validation 
  [sadness ] im feeling quite sad and sorry for myself but ill snap out of it soon
  [sadness ] i feel like i am still looking at a blank canvas blank pieces of paper
  [love    ] i feel like a faithful servant


В датасете классифицируется эмоциональная окраска твита, насторение автора.

## Токенизация

In [9]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example_text = dataset['train'][1]['text']

In [10]:
encoding = tokenizer(
    example_text,
    add_special_tokens=True,
    return_attention_mask=True,
    return_special_tokens_mask=True,  
)

print("Исходный текст:")
print(example_text)
print()

print("Токены (с спец-токенами):")
print(tokenizer.convert_ids_to_tokens(encoding["input_ids"]))
print()

print("input_ids:")
print(encoding["input_ids"])
print()

print("attention_mask:")
print(encoding["attention_mask"])
print()

print("special_tokens_mask (1 = спец-токен):")
print(encoding["special_tokens_mask"])

Исходный текст:
i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

Токены (с спец-токенами):
['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake', '[SEP]']

input_ids:
[101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998, 2003, 8300, 102]

attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

special_tokens_mask (1 = спец-токен):
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]


In [11]:
def inspect_single_text(text: str, tokenizer) -> pd.DataFrame:
    encoding = tokenizer(
        text,
        add_special_tokens=True,
        return_attention_mask=True,
        return_special_tokens_mask=True,
    )
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])
    return pd.DataFrame({
        "position":            list(range(len(tokens))),
        "token":               tokens,
        "token_id":            encoding["input_ids"],
        "attention_mask":      encoding["attention_mask"],
        "is_special_token":    encoding["special_tokens_mask"],
    })

for i in range(3):
    ex = dataset['train'][i]
    label_str = dataset['train'].features['label'].names[ex['label']]
    print(f"\n[{label_str}] {ex['text']}")
    df = inspect_single_text(ex['text'], tokenizer)
    display(df)


[sadness] i didnt feel humiliated


,position,token,token_id,attention_mask,is_special_token
0,0,[CLS],101,1,1
1,1,i,1045,1,0
2,2,didn,2134,1,0
3,3,##t,2102,1,0
4,4,feel,2514,1,0
5,5,humiliated,26608,1,0
6,6,[SEP],102,1,1



[sadness] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake


,position,token,token_id,attention_mask,is_special_token
0,0,[CLS],101,1,1
1,1,i,1045,1,0
2,2,can,2064,1,0
3,3,go,2175,1,0
4,4,from,2013,1,0
5,5,feeling,3110,1,0
6,6,so,2061,1,0
7,7,hopeless,20625,1,0
8,8,to,2000,1,0
9,9,so,2061,1,0



[anger] im grabbing a minute to post i feel greedy wrong


,position,token,token_id,attention_mask,is_special_token
0,0,[CLS],101,1,1
1,1,im,10047,1,0
2,2,grabbing,9775,1,0
3,3,a,1037,1,0
4,4,minute,3371,1,0
5,5,to,2000,1,0
6,6,post,2695,1,0
7,7,i,1045,1,0
8,8,feel,2514,1,0
9,9,greedy,20505,1,0


In [12]:
short_text  = "I feel happy"
medium_text = dataset['train'][0]['text']
long_text   = " ".join(["word"] * 100)  # искусственно длинный текст

texts = [short_text, medium_text, long_text]

batch_padded = tokenizer(
    texts,
    padding="longest",       # все выравниваются до самого длинного
    truncation=False,
    return_tensors="pt",
)
print("=== padding='longest', truncation=False ===")
print("input_ids shape:", batch_padded["input_ids"].shape)
for i, text in enumerate(texts):
    ids   = batch_padded["input_ids"][i].tolist()
    mask  = batch_padded["attention_mask"][i].tolist()
    n_pad = mask.count(0)
    print(f"  Текст {i+1}: длина={len(ids)}, PAD-токенов={n_pad}")

print()

batch_trunc = tokenizer(
    texts,
    padding="max_length",    # паддинг до max_length
    truncation=True,         # обрезаем всё длиннее max_length
    max_length=16,
    return_tensors="pt",
)
print("=== padding='max_length', truncation=True, max_length=16 ===")
print("input_ids shape:", batch_trunc["input_ids"].shape)
for i, text in enumerate(texts):
    ids  = batch_trunc["input_ids"][i].tolist()
    mask = batch_trunc["attention_mask"][i].tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    print(f"  Текст {i+1}: {toks}")
    print(f"           mask: {mask}")
    print()

=== padding='longest', truncation=False ===
input_ids shape: torch.Size([3, 102])
  Текст 1: длина=102, PAD-токенов=97
  Текст 2: длина=102, PAD-токенов=95
  Текст 3: длина=102, PAD-токенов=0

=== padding='max_length', truncation=True, max_length=16 ===
input_ids shape: torch.Size([3, 16])
  Текст 1: ['[CLS]', 'i', 'feel', 'happy', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
           mask: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

  Текст 2: ['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
           mask: [1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

  Текст 3: ['[CLS]', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', '[SEP]']
           mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]



In [13]:
print("Special tokens:")
print(f"  [CLS] token : '{tokenizer.cls_token}'  id={tokenizer.cls_token_id}")
print(f"  [SEP] token : '{tokenizer.sep_token}'  id={tokenizer.sep_token_id}")
print(f"  [PAD] token : '{tokenizer.pad_token}'  id={tokenizer.pad_token_id}")
print(f"  [UNK] token : '{tokenizer.unk_token}'  id={tokenizer.unk_token_id}")
print(f"  [MASK] token: '{tokenizer.mask_token}' id={tokenizer.mask_token_id}")
print()

# Покажем, как CLS и SEP оборачивают текст
text = "I feel great today"
tokens_no_special = tokenizer.tokenize(text)
tokens_with_special = tokenizer.convert_ids_to_tokens(
    tokenizer.encode(text, add_special_tokens=True)
)
print(f"Без спец-токенов : {tokens_no_special}")
print(f"Со спец-токенами : {tokens_with_special}")
#  --> ['[CLS]', 'i', 'feel', 'great', 'today', '[SEP]']

Special tokens:
  [CLS] token : '[CLS]'  id=101
  [SEP] token : '[SEP]'  id=102
  [PAD] token : '[PAD]'  id=0
  [UNK] token : '[UNK]'  id=100
  [MASK] token: '[MASK]' id=103

Без спец-токенов : ['i', 'feel', 'great', 'today']
Со спец-токенами : ['[CLS]', 'i', 'feel', 'great', 'today', '[SEP]']


## Инференс готовой модели

In [14]:
classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    return_all_scores=False,   
    device=device 
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 35002.54it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [21]:
print(f"Классы нашего датасета : {label_names}")
print(f"Классы pretrained модели: anger, disgust, fear, joy, neutral, sadness, surprise")
print()

# Берём 5 примеров из test-части датасета
test_examples = dataset["test"].select(range(5))

print(f"{'Текст (обрезан до 60 символов)':<62} {'Истина':<10} {'Предсказание':<12} {'Score'}")
print("-" * 100)

for ex in test_examples:
    text        = ex["text"]
    true_label  = label_names[ex["label"]]
    result      = classifier(text)[0]
    pred_label  = result["label"]
    score       = result["score"]
    match       = "✅" if pred_label == true_label else "❌"
    print(f"{text[:60]:<62} {true_label:<10} {pred_label:<12} {score:.2f}  {match}")

Классы нашего датасета : ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Классы pretrained модели: anger, disgust, fear, joy, neutral, sadness, surprise

Текст (обрезан до 60 символов)                                 Истина     Предсказание Score
----------------------------------------------------------------------------------------------------
im feeling rather rotten so im not very ambitious right now    sadness    sadness      0.99  ✅
im updating my blog because i feel shitty                      sadness    sadness      0.99  ✅
i never make her separate from me because i don t ever want    sadness    sadness      0.98  ✅
i left with my bouquet of red and yellow tulips under my arm   joy        joy          0.99  ✅
i was feeling a little vain when i did this one                sadness    sadness      0.99  ✅


In [22]:
print("\n" + "=" * 70)
print("ПОДРОБНЫЙ РАЗБОР 5 ПРИМЕРОВ")
print("=" * 70)

classifier_full = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=device
)

for i, ex in enumerate(test_examples):
    text       = ex["text"]
    true_label = label_names[ex["label"]]
    
    # top_k=None → pipeline возвращает [[{...}, {...}, ...]]
    # [0] берёт результат для первого (единственного) текста
    all_scores = classifier_full(text)[0]  # список словарей

    print(f"\nПример {i+1}:")
    print(f"  Текст         : {text}")
    print(f"  Истинная метка: {true_label}")
    print(f"  Предсказания  :")
    for item in sorted(all_scores, key=lambda x: x["score"], reverse=True):
        bar = "█" * int(item["score"] * 20)
        print(f"    {item['label']:10s}: {item['score']:.3f}  {bar}")


ПОДРОБНЫЙ РАЗБОР 5 ПРИМЕРОВ


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 34999.76it/s]
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Пример 1:
  Текст         : im feeling rather rotten so im not very ambitious right now
  Истинная метка: sadness
  Предсказания  :
    sadness   : 0.992  ███████████████████
    anger     : 0.003  
    joy       : 0.003  
    surprise  : 0.001  
    disgust   : 0.001  
    fear      : 0.001  
    neutral   : 0.000  

Пример 2:
  Текст         : im updating my blog because i feel shitty
  Истинная метка: sadness
  Предсказания  :
    sadness   : 0.993  ███████████████████
    anger     : 0.001  
    disgust   : 0.001  
    neutral   : 0.001  
    fear      : 0.001  
    surprise  : 0.001  
    joy       : 0.001  

Пример 3:
  Текст         : i never make her separate from me because i don t ever want her to feel like i m ashamed with her
  Истинная метка: sadness
  Предсказания  :
    sadness   : 0.982  ███████████████████
    neutral   : 0.005  
    disgust   : 0.005  
    surprise  : 0.003  
    fear      : 0.002  
    joy       : 0.002  
    anger     : 0.001  

Пример 4:
  Текст  

In [23]:
print("\n" + "=" * 70)
print("АНАЛИЗ ПРИГОДНОСТИ PRETRAINED МОДЕЛИ")
print("=" * 70)

our_classes     = set(label_names)
model_classes   = {"anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"}

only_in_ours    = our_classes - model_classes
only_in_model   = model_classes - our_classes
overlap         = our_classes & model_classes

print(f"Наши классы    : {sorted(our_classes)}")
print(f"Классы модели  : {sorted(model_classes)}")
print(f"Совпадают      : {sorted(overlap)}   ({len(overlap)}/{len(our_classes)})")
print(f"Только у нас   : {only_in_ours}   ← модель их не предсказывает!")
print(f"Только в модели: {only_in_model}  ← лишние классы")


АНАЛИЗ ПРИГОДНОСТИ PRETRAINED МОДЕЛИ
Наши классы    : ['anger', 'fear', 'joy', 'love', 'sadness', 'surprise']
Классы модели  : ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
Совпадают      : ['anger', 'fear', 'joy', 'sadness', 'surprise']   (5/6)
Только у нас   : {'love'}   ← модель их не предсказывает!
Только в модели: {'neutral', 'disgust'}  ← лишние классы


## Fine-tuning для классификации текста

## Оценка качества

In [15]:
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions    = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1  = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="weighted"   # взвешенный F1 для несбалансированных классов
    )
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

NameError: name 'evaluate' is not defined